# STARK-Amazon Data Exploration — QA Dataset

Inspects splits, queries, and corpus statistics using only the QA dataset.
**Runs on Mac M2.** Does not load the SKB (which requires ~19 GB RAM).
See `01b_data_exploration_skb.ipynb` for product-node and graph-edge inspection (GWDG HPC only).

## 1. Load QA dataset

In [1]:
from stark_qa import load_qa

qa_dataset = load_qa('amazon')
print(f'QA dataset size: {len(qa_dataset)}')

/Users/shreeshangaavin/ShreeShangaavi/master thesis/.venv/lib/python3.12/site-packages/outdated/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Use file from /Users/shreeshangaavin/.cache/huggingface/hub/datasets--snap-stanford--stark/snapshots/88269e23e90587f99476c5dd74e235a0877e69be/qa/amazon/stark_qa/stark_qa_human_generated_eval.csv.
QA dataset size: 9100


## 2. Inspect the official splits

In [2]:
idx_split = qa_dataset.get_idx_split()
print('Available splits:', list(idx_split.keys()))

for split_name, ids in idx_split.items():
    print(f'  {split_name:12s}: {len(ids):,} queries')

Available splits: ['train', 'val', 'test', 'test-0.1']
  train       : 5,910 queries
  val         : 1,548 queries
  test        : 1,642 queries
  test-0.1    : 164 queries


In [8]:
test_01_ids_int = [int(i) for i in test_01_ids]
test_ids_int = [int(i) for i in test_ids]

assert set(test_01_ids_int).issubset(set(test_ids_int)), 'test-0.1 is NOT a subset of test!'
print('Confirmed: test-0.1 is a subset of test')

Confirmed: test-0.1 is a subset of test


Debug test-0.1 set

In [4]:
# Find which indices are in test-0.1 but not in test
not_in_test = set(test_01_ids) - set(test_ids)
print(f'Indices in test-0.1 but NOT in test: {len(not_in_test)}')
print(f'Sample: {list(not_in_test)[:10]}')

# Check what split those indices belong to
for idx in list(not_in_test)[:5]:
    query, answer_ids, _ = qa_dataset[idx]
    print(f'idx={idx} | {query[:80]}')

Indices in test-0.1 but NOT in test: 164
Sample: [tensor(214), tensor(2759), tensor(8277), tensor(4942), tensor(6753), tensor(1815), tensor(3472), tensor(5915), tensor(7328), tensor(8900)]


ValueError: too many values to unpack (expected 3)

In [5]:
# Check what qa_dataset actually returns
sample = qa_dataset[list(not_in_test)[0]]
print(f'Type: {type(sample)}')
print(f'Length: {len(sample)}')
print(sample)

Type: <class 'tuple'>
Length: 4
('Does anyone know of a special pack for NEFC enthusiasts that includes a decorative sticker?', 214, [599498, 704331, 153100, 808074, 3118, 193392, 110261], None)


In [6]:
# Find which indices are in test-0.1 but not in test
not_in_test = set(test_01_ids) - set(test_ids)
print(f'Indices in test-0.1 but NOT in test: {len(not_in_test)}')
print(f'Sample: {list(not_in_test)[:10]}')

# Check what those indices look like
for idx in list(not_in_test)[:5]:
    query, query_id, answer_ids, meta = qa_dataset[idx]
    print(f'idx={idx} | query_id={query_id} | {query[:80]}')

# Also check overlap the other way
in_both = set(test_01_ids) & set(test_ids)
print(f'\nIndices in BOTH test and test-0.1: {len(in_both)}')
print(f'Total test-0.1 size: {len(test_01_ids)}')

Indices in test-0.1 but NOT in test: 164
Sample: [tensor(788), tensor(2992), tensor(7896), tensor(4614), tensor(8426), tensor(1436), tensor(2154), tensor(6246), tensor(6753), tensor(1991)]
idx=788 | query_id=788 | I have been using a Satori SOLO Bike Bicycle Suspension Seatpost 27.2x350mm. Can
idx=2992 | query_id=2992 | Looking for a high-quality leather pouch bag with medieval aesthetics for the up
idx=7896 | query_id=7896 | What's a good compact, all-purpose skateboard tool that's compatible with both p
idx=4614 | query_id=4614 | Where can I find a durable, long-lasting barrel sock or cover from Wicked Sports
idx=8426 | query_id=8426 | Looking for suggestions for silicone wedding ring sets for men, preferably in ma

Indices in BOTH test and test-0.1: 0
Total test-0.1 size: 164


In [7]:
# Convert tensor indices to plain ints and re-check
test_01_ids_int = [int(i) for i in test_01_ids]
test_ids_int = [int(i) for i in test_ids]

not_in_test = set(test_01_ids_int) - set(test_ids_int)
in_both = set(test_01_ids_int) & set(test_ids_int)

print(f'After int conversion:')
print(f'  Indices in test-0.1 but NOT in test: {len(not_in_test)}')
print(f'  Indices in BOTH: {len(in_both)}')

After int conversion:
  Indices in test-0.1 but NOT in test: 0
  Indices in BOTH: 164


End Debug


## 3. Inspect a single query

In [10]:
test_01_ids_int = [int(i) for i in idx_split['test-0.1']]

for i in range(5):
    idx = test_01_ids_int[i]
    query, query_id, answer_ids, meta = qa_dataset[idx]
    suffix = '...' if len(answer_ids) > 5 else ''
    print(f'[{i}] query_id={query_id}')
    print(f'     {query}')
    print(f'     answers: {answer_ids[:5]}{suffix}')
    print()

[0] query_id=3
     Is there a durable, waterproof trail map available for hiking and biking that can withstand rainy conditions?
     answers: [130947, 629636, 68, 38854, 73]...

[1] query_id=85
     What are some recommended rubber overgrips for sports equipment?
     answers: [57540, 749990, 293064, 56393, 285005]...

[2] query_id=135
     What's the best scope mount for a Colt 1911 and Colt Government 45 that still allows for iron sights usage and optics mounting? Also, I need it to be compatible with non-ambidextrous safety. Suggestions?
     answers: [1976, 170902, 583740, 583742]

[3] query_id=173
     Can you recommend a Thill brand fishing bobber? I specifically prefer this brand.
     answers: [2532]

[4] query_id=214
     Does anyone know of a special pack for NEFC enthusiasts that includes a decorative sticker?
     answers: [599498, 704331, 153100, 808074, 3118]...



## 4. Quick corpus statistics

In [12]:
import numpy as np

test_ids_int = [int(i) for i in idx_split['test']]

# Distribution of answer set sizes across the test split
answer_sizes = [len(qa_dataset[i][2]) for i in test_ids_int]
print(f'Answer set size — min: {min(answer_sizes)}, max: {max(answer_sizes)}, '
      f'mean: {np.mean(answer_sizes):.1f}, median: {np.median(answer_sizes):.0f}')

# Query length distribution (word count)
query_lengths = [len(qa_dataset[i][0].split()) for i in test_ids_int]
print(f'Query length (words) — min: {min(query_lengths)}, max: {max(query_lengths)}, '
      f'mean: {np.mean(query_lengths):.1f}')

Answer set size — min: 1, max: 20, mean: 5.4, median: 3
Query length (words) — min: 7, max: 85, mean: 24.6
